# 🌱 Relatório Estatístico — Registro Nacional de Cultivares (RNC)

> **Fonte de dados:** `relatorio_cultivares.csv`  
> **Filtro Aplicado:** Excluindo registros de 2026

---

## Seções
1. [Carregamento e Pré-processamento](#1-carregamento)
2. [Análise de Valores Nulos](#2-nulos)
3. [Cultivares Mais Comuns](#3-cultivares)
4. [Crescimento Anual de Registros](#4-crescimento-anual)
5. [Classificação de Mantenedores por Setor](#5-setores)
6. [Setor Público — Detalhamento](#6-publico)
7. [Setor Misto — Detalhamento](#7-misto)
8. [Líderes: Soja e Milho](#8-soja-milho)
9. [Mantenedores nas Cultivares Mais Comuns](#9-lideres)

---
## 1. Carregamento e Pré-processamento <a id='1-carregamento'></a>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 130, 'figure.figsize': (12, 5)})

COL_DATA       = 'DATA DO REGISTRO'
COL_MANTENEDOR = 'MANTENEDOR (REQUERENTE) (NOME)'
COL_NOME_COM   = 'NOME COMUM'
COL_FORMULARIO = 'Nº FORMULÁRIO'

df = pd.read_csv('relatorio_cultivares.csv')
df[COL_DATA] = pd.to_datetime(df[COL_DATA], dayfirst=True, errors='coerce')
df['ANO'] = df[COL_DATA].dt.year
df = df[df['ANO'] != 2026]

PUB_KW = ['universidade', 'empresa brasileira', 'instituto', 'empresa de pesquisa agropecuária', 'departamento de sementes', 'secretaria da agricultura', 'agência goiana']
MISTO_KW = ['fundação', 'cooperativa']

def classify_setor(nome):
    if pd.isna(nome): return 'Nulo'
    n = nome.lower()
    for kw in PUB_KW: 
        if kw in n: return 'Público'
    for kw in MISTO_KW: 
        if kw in n: return 'Misto'
    return 'Privado'

df['SETOR'] = df[COL_MANTENEDOR].apply(classify_setor)

---
## 2. Análise de Valores Nulos <a id='2-nulos'></a>

In [ ]:
def analise_nulos(df):
    nulos = df.isnull().sum()
    nulos = nulos[nulos > 0].sort_values(ascending=False).to_frame(name='Total Nulos')
    nulos['% Nulos'] = (nulos['Total Nulos'] / len(df) * 100).round(2)
    
    print(f"📊 Total de registros no dataset: {len(df):,}")
    print(f"❌ Nulos em Nº FORMULÁRIO: {df[COL_FORMULARIO].isnull().sum():,}")
    print(f"❌ Nulos em MANTENEDOR:    {df[COL_MANTENEDOR].isnull().sum():,}")
    
    return nulos.style.background_gradient(cmap='Reds')

analise_nulos(df)

---
## 3. Cultivares Mais Comuns <a id='3-cultivares'></a>

In [ ]:
top_cultivares = df.groupby(COL_NOME_COM).agg(Registros=(COL_NOME_COM, 'count')).reset_index().sort_values('Registros', ascending=False).head(30)
top_cultivares.index = range(1, 31)
display(top_cultivares.style.background_gradient(subset=['Registros'], cmap='YlGn'))

---
## 4. Crescimento Anual de Registros <a id='4-crescimento-anual'></a>

In [ ]:
reg_ano = df.groupby('ANO').size().reset_index(name='Registros').set_index('ANO')
fig, ax = plt.subplots(); ax.bar(reg_ano.index.astype(int), reg_ano['Registros'], color='#2E7D32'); plt.show()

---
## 5. Classificação de Mantenedores por Setor <a id='5-setores'></a>

In [ ]:
dist = df['SETOR'].value_counts().to_frame(); display(dist.style.background_gradient(cmap='Blues'))

---
## 6. Setor Público — Detalhamento <a id='6-publico'></a>

In [ ]:
df_pub = df[df['SETOR'] == 'Público'].copy()
top_pub = df_pub[COL_MANTENEDOR].value_counts().head(20).reset_index()
top_pub.index = range(1, 21)
display(top_pub.style.background_gradient(cmap='Blues'))

---
## 7. Setor Misto — Detalhamento <a id='7-misto'></a>

In [ ]:
display(df[df['SETOR'] == 'Misto'][COL_MANTENEDOR].value_counts().head(20).reset_index().style.background_gradient(cmap='YlOrBr'))

---
## 8. Líderes: Soja e Milho <a id='8-soja-milho'></a>

In [ ]:
top5_soja = df[df[COL_NOME_COM].str.upper().str.contains('SOJA', na=False)][COL_MANTENEDOR].value_counts().head(5).reset_index()
top5_soja.columns = ['Mantenedor Soja', 'Registros']
top5_soja.index = range(1, 6)

top5_milho = df[df[COL_NOME_COM].str.upper().str.contains('MILHO', na=False)][COL_MANTENEDOR].value_counts().head(5).reset_index()
top5_milho.columns = ['Mantenedor Milho', 'Registros']
top5_milho.index = range(1, 6)

print("🟢 TOP 5 MANTENEDORES DE SOJA")
display(top5_soja.style.background_gradient(cmap='Greens'))
print("\n🟡 TOP 5 MANTENEDORES DE MILHO")
display(top5_milho.style.background_gradient(cmap='Oranges'))

---
## 9. Mantenedores nas Cultivares Mais Comuns <a id='10-lideres'></a>

In [ ]:
pop_30 = df[COL_NOME_COM].value_counts().head(30).index
t_mant = df[df[COL_NOME_COM].isin(pop_30)][COL_MANTENEDOR].value_counts().head(20).reset_index()
t_mant.index = range(1, 21)
display(t_mant.style.background_gradient(cmap='Purples'))